# T3 14-DOF chassis EOM — SymPy Kane's-method derivation

Authoritative symbolic source for the T3 transient chassis equations of motion (Decision #32,
HANDOFF §11.6). This notebook extends the T2 7-DOF planar chassis (`t2_chassis_kane.ipynb`) to the
full 14-DOF tier: sprung heave/pitch/roll + four unsprung verticals + nonlinear suspension
(spring / bump-rebound damper / C¹ bumpstop / anti-roll bar) + tyre vertical springs, plus the two
refinement terms the T2 derivation flagged out of scope (user-locked to land here): the gyroscopic
wheel spin×yaw coupling and the 3-D frame-transport vertical-curvature term. It assembles the full
24-state right-hand side and regenerates the committed fixture `fixtures/t3_chassis_rhs.json` that
the Rust test `outlap-vehicle/tests/kane_fixture_t3.rs` checks to 1e-12. The heavy lifting lives in
the importable module `t3_chassis_kane.py`.

**Do not hand-edit the fixture** — re-run this notebook. CI re-executes it and `git diff --exit-code`s
the fixture, so the symbolic derivation stays the single source of truth for the EOM signs.

References: Milliken & Milliken *Race Car Vehicle Dynamics* (load transfer, roll-centre/anti,
K&C); Guiggiani *The Science of Vehicle Dynamics* (14-DOF ride/handling, roll axis); Pacejka 2012
ch. 1 (tyre vertical stiffness); Kane & Levinson (method). See `docs/theory/t3-chassis.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import t3_chassis_kane as k

# Build + lambdify the 24-state RHS. `build_rhs` pins the sign conventions in its docstring; the
# reduction checks live in the Rust static-equilibrium / restoring tests.
_fn, arg_names = k.build_rhs_lambda()
print("T3 RHS states:", len(k.STATE_ORDER), "| arguments:", len(arg_names))

In [ ]:
# Regenerate the committed fixture (seeded; deterministic).
count = k.generate()
print("wrote", count, "random-state RHS samples to", k.FIXTURE)